#### Conexão com o Neon 

Este notebook só cuida da conexão com o banco. Os outros notebooks do
pipeline usam `%run "./apollo_connection_neon_core"` no início pra
reaproveitar as views criadas aqui, sem precisar repetir credencial em
lugar nenhum.

**Pré-requisito:** secret scope `apollo` já criado, com as chaves
`neon_user` e `neon_password`.

In [0]:
NEON_HOST = "ep-snowy-boat-ac8ly9cb.sa-east-1.aws.neon.tech"
NEON_PORT = "5432"
NEON_DATABASE = "neondb"

#### Monta a URL e busca a senha do Secret Scope

In [0]:
try:
    neon_user = dbutils.secrets.get(scope="apollo", key="neon_user")
    neon_password = dbutils.secrets.get(scope="apollo", key="neon_password")
    print("Segredos lidos com sucesso do scope 'apollo'.")
except Exception as e:
    raise Exception(
        "Não consegui ler o secret scope 'apollo'. Confirme se ele existe "
        "(databricks secrets list-scopes) e se as chaves neon_user e "
        f"neon_password foram criadas. Erro original: {e}"
    )

jdbc_url = f"jdbc:postgresql://{NEON_HOST}:{NEON_PORT}/{NEON_DATABASE}?sslmode=require"

connection_properties = {
    "user": neon_user,
    "password": neon_password,
    "driver": "org.postgresql.Driver",
}

#### Teste rápido de conexão

In [0]:
try:
    df_teste = spark.read.jdbc(url=jdbc_url, table="company", properties=connection_properties)
    print(f"Conexão OK. {df_teste.count()} linha(s) encontrada(s) em 'company'.")
    df_teste.show(5, truncate=False)
except Exception as e:
    raise Exception(
        "Falha ao conectar no Neon. Causas mais comuns:\n"
        "1) Host/porta/nome do banco errados na Célula 1\n"
        "2) Usuário ou senha errados no secret scope\n"
        "3) Bloqueio de rede de saída do Databricks Free Edition "
        "(o domínio do Neon pode não estar liberado)\n"
        f"Erro original: {e}"
    )

#### Views usadas pelo Pipeline

In [0]:
panels_batch_query = """
    (SELECT
        p.id AS panel_id,
        p.batch_id,
        p.estimated_life_cycle,
        p.rated_efficiency,
        p.activated_at,
        b.company_unit_id,
        b.unit_cost,
        b.panels_qtt
     FROM using_panels p
     JOIN batch b ON b.id = p.batch_id
     WHERE p.activated_at IS NOT NULL) AS panels_batch
"""

try:
    df_panels_batch = spark.read.jdbc(url=jdbc_url, table=panels_batch_query, properties=connection_properties)
    df_panels_batch.createOrReplaceTempView("stg_panels_batch")

    df_measurements = spark.read.jdbc(url=jdbc_url, table="panel_performance_measurement", properties=connection_properties)
    df_measurements.createOrReplaceTempView("stg_measurements")

    df_company_unit = spark.read.jdbc(url=jdbc_url, table="dm.dim_company_unit", properties=connection_properties)
    df_company_unit.createOrReplaceTempView("stg_company_unit")

    print("Views criadas: stg_panels_batch, stg_measurements, stg_company_unit")
except Exception as e:
    raise Exception(f"Falha ao criar as views de leitura. Erro original: {e}")

#### Tabela de funcionários (base do Row Filter por filial)

In [0]:
employee_query = """
    (SELECT email, company_unit_id
     FROM employee
     WHERE is_active = true) AS employee_filial
"""
 
try:
    df_employee = spark.read.jdbc(url=jdbc_url, table=employee_query, properties=connection_properties)
    (
        df_employee.write
        .format("delta")
        .mode("overwrite")
        .saveAsTable("apollo_bi.dim_employee")
    )
    print(f"apollo_bi.dim_employee atualizada: {df_employee.count()} funcionário(s) ativo(s).")
except Exception as e:
    raise Exception(f"Falha ao atualizar apollo_bi.dim_employee. Erro original: {e}")

#### Checagem final

In [0]:
for view_name in ["stg_panels_batch", "stg_measurements", "stg_company_unit"]:
    count = spark.sql(f"SELECT COUNT(*) AS c FROM {view_name}").collect()[0]["c"]
    print(f"{view_name}: {count} linha(s)")